#Schritt 6: Playlist-Aggregation

In [ ]:
import pandas as pd
import numpy as np

MINDEST_ANZAHL_SONGS = 5

In [ ]:
def zähle_risiko_kategorien_pro_playlist(daten):
    tabelle = pd.crosstab(daten['playlist_id'], daten['risiko_label'])
    return tabelle

In [ ]:
def berechne_playlist_profil(daten, mindest_anzahl_songs=MINDEST_ANZAHL_SONGS):
    profil_liste = []
    alle_playlists = daten['playlist_id'].unique()
    for playlist_id in alle_playlists:
        playlist_daten = daten[daten['playlist_id'] == playlist_id]
        anzahl_songs = len(playlist_daten)
        anzahl_risikohinweis = len(playlist_daten[playlist_daten['risiko_label'] == 'möglicher_risikohinweis'])
        anzahl_herzschmerz = len(playlist_daten[playlist_daten['risiko_label'] == 'herzschmerz'])
        anzahl_melancholie = len(playlist_daten[playlist_daten['risiko_label'] == 'ästhetische_melancholie'])
        anzahl_kein_hinweis = len(playlist_daten[playlist_daten['risiko_label'] == 'kein_hinweis'])
        anteil_risikohinweis_prozent = anzahl_risikohinweis / anzahl_songs * 100
        zu_klein_für_aussage = anzahl_songs < mindest_anzahl_songs
        profil_liste.append({'playlist_id': playlist_id, 'anzahl_songs': anzahl_songs, 'zu_klein_für_verlaessliche_aussage': zu_klein_für_aussage, 'anzahl_möglicher_risikohinweis': anzahl_risikohinweis, 'anzahl_herzschmerz': anzahl_herzschmerz, 'anzahl_ästhetische_melancholie': anzahl_melancholie, 'anzahl_kein_hinweis': anzahl_kein_hinweis, 'anteil_risikohinweis_prozent': round(anteil_risikohinweis_prozent, 1)})
    profil_tabelle = pd.DataFrame(profil_liste)
    return profil_tabelle

In [ ]:
def simuliere_playlist_grössen(daten, grössen=(2, 5, 10, 20, 30), anzahl_wiederholungen=200, zufalls_seed=42):
    rng = np.random.default_rng(zufalls_seed)
    alle_labels = daten['risiko_label'].values
    ergebnisse = []
    for grösse in grössen:
        anteile = []
        for _ in range(anzahl_wiederholungen):
            stichprobe = rng.choice(alle_labels, size=grösse, replace=True)
            anteil = (stichprobe == 'möglicher_risikohinweis').mean() * 100
            anteile.append(anteil)
        ergebnisse.append({
            'playlist_grösse': grösse,
            'mittlerer_anteil_risikohinweis_prozent': round(float(np.mean(anteile)), 1),
            'standardabweichung_prozent': round(float(np.std(anteile)), 1),
        })
    return pd.DataFrame(ergebnisse)

#Pipeline ausführen

In [ ]:
daten = pd.read_csv('data/daten_mit_risiko.csv')
kreuztabelle = zähle_risiko_kategorien_pro_playlist(daten)
print('Kreuztabelle (Songs pro Playlist und Kategorie):')
print(kreuztabelle)
print()
profil = berechne_playlist_profil(daten)
print(f'Playlist-Profile (Mindestgrösse fuer eine Aussage: {MINDEST_ANZAHL_SONGS} Songs):')
print(profil)
print()
print('Sensitivitätsanalyse: wie verlässlich ist der Anteil je nach Playlist-Grösse?')
sensitivität = simuliere_playlist_grössen(daten)
print(sensitivität)
print()
print('Je kleiner die Playlist, desto grösser die Standardabweichung')

In [ ]:
zu_kleine_playlists = profil[profil['zu_klein_für_verlässliche_aussage']]
if len(zu_kleine_playlists) > 0:
    print(f'\nFolgende Playlists haben weniger als {MINDEST_ANZAHL_SONGS} Songs. Der berechnete Prozentwert ist bei so wenigen Songs nicht verlässlich.')
    print(zu_kleine_playlists['playlist_id'].tolist())
profil.to_csv('data/playlist_profile.csv', index=False)
sensitivitaet.to_csv('data/playlist_groessen_sensitivität.csv', index=False)
print('\nPlaylist-Profile gespeichert unter: data/playlist_profile.csv')
print('Sensitivitätsanalyse gespeichert unter: data/playlist_groessen_sensitivität.csv')